In [1]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')
print("Working directory:", os.getcwd())

Working directory: C:\Users\Klara\retail-intelligence


In [2]:
import pandas as pd
import numpy as np
import shap
import joblib
import sys
sys.path.append('.')

from src.module1_churn.data_preparation import prepare_churn_data
from src.module1_churn.shap_explainability import (
    compute_shap_values, 
    explain_single_customer,
    generate_natural_language_explanation
)

data = prepare_churn_data()
X_test = data['X_test']
y_test = data['y_test']

model = joblib.load('models/churn_model_tuned.pkl')
optimal_threshold = float(np.load('models/optimal_threshold.npy'))
y_prob = model.predict_proba(X_test)[:, 1]

explainer = joblib.load('models/shap_explainer.pkl')
shap_values = explainer(X_test)

Loading data...

📊 Data preparation complete:
   Training set: 3,136 customers
   Test set: 784 customers
   Features: 7

   Training churn rate: 33.4%
   Test churn rate: 33.3%


In [3]:
high_risk_idx = int(np.argmax(y_prob))
low_risk_idx = int(np.argmin(y_prob))
borderline_idx = int(np.argmin(np.abs(y_prob - optimal_threshold)))

print("Testing explanations on 3 customer profiles:")
print(f"High risk customer index: {high_risk_idx}")
print(f"Low risk customer index: {low_risk_idx}")
print(f"Borderline customer index: {borderline_idx}")

for idx, label in [(high_risk_idx, "High Risk"), 
                    (low_risk_idx, "Low Risk"),
                    (borderline_idx, "Borderline")]:
    print(f"\n{'='*55}")
    print(f"Testing: {label} Customer")
    explanation = explain_single_customer(
        idx, X_test, shap_values, y_prob, optimal_threshold
    )
    nl_explanation = generate_natural_language_explanation(explanation, optimal_threshold)
    print(f"\n📝 Natural Language Explanation:")
    print(f"   {nl_explanation}")

Testing explanations on 3 customer profiles:
High risk customer index: 178
Low risk customer index: 0
Borderline customer index: 299

Testing: High Risk Customer

Customer 178 Analysis
Churn Probability: 80.2%
Prediction: ⚠️ HIGH RISK

🔴 Top factors INCREASING churn risk:
   DaysActive: 0.00 (SHAP: +0.5741)
   UniqueProducts: 6.00 (SHAP: +0.3662)
   AvgQuantity: 3.00 (SHAP: +0.3574)

🟢 Top factors DECREASING churn risk:
   OrdersPerDay: 1.00 (SHAP: -0.0255)
   AvgOrderValue: 119.30 (SHAP: -0.0014)

📝 Natural Language Explanation:
   This customer has a extreme churn risk (80.2%). The primary driver is their DaysActive (value: 0.0), which increases churn likelihood. Additionally, their UniqueProducts (value: 6.0) contributes to this risk.

Testing: Low Risk Customer

Customer 0 Analysis
Churn Probability: 7.0%
Prediction: ✅ LOW RISK

🔴 Top factors INCREASING churn risk:
   OrdersPerDay: 0.01 (SHAP: +0.0192)

🟢 Top factors DECREASING churn risk:
   DaysActive: 323.00 (SHAP: -2.2779)
   U

In [4]:
import random
random.seed(42)
random_indices = random.sample(range(len(X_test)), 3)

for idx in random_indices:
    print(f"\n{'='*55}")
    print(f"Testing: Random Customer {idx}")
    explanation = explain_single_customer(
        idx, X_test, shap_values, y_prob, optimal_threshold
    )
    nl_explanation = generate_natural_language_explanation(explanation, optimal_threshold)
    print(f"\n📝 Natural Language Explanation:")
    print(f"   {nl_explanation}")


Testing: Random Customer 654

Customer 654 Analysis
Churn Probability: 64.2%
Prediction: ⚠️ HIGH RISK

🔴 Top factors INCREASING churn risk:
   DaysActive: 1.00 (SHAP: +0.5109)
   UniqueProducts: 23.00 (SHAP: +0.1677)
   Monetary: 429.60 (SHAP: +0.0755)

🟢 Top factors DECREASING churn risk:
   AvgQuantity: 15.26 (SHAP: -0.1511)
   OrdersPerDay: 2.00 (SHAP: -0.0314)
   AvgOrderValue: 214.80 (SHAP: -0.0110)

📝 Natural Language Explanation:
   This customer has a high churn risk (64.2%). The primary driver is their DaysActive (value: 1.0), which increases churn likelihood. Additionally, their UniqueProducts (value: 23.0) contributes to this risk. This is partially offset by their AvgQuantity (value: 15.3), which supports retention.

Testing: Random Customer 114

Customer 114 Analysis
Churn Probability: 37.7%
Prediction: ⚠️ HIGH RISK

🔴 Top factors INCREASING churn risk:
   DaysActive: 192.00 (SHAP: +0.2797)
   AvgQuantity: 2.31 (SHAP: +0.2249)

🟢 Top factors DECREASING churn risk:
   Uniq

In [5]:
extreme_candidates = np.where(y_prob >= 0.70)[0]
extreme_idx = int(extreme_candidates[0])

print(f"\n{'='*55}")
print(f"Testing: Extreme Risk Customer")
explanation = explain_single_customer(
    extreme_idx, X_test, shap_values, y_prob, optimal_threshold
)
nl_explanation = generate_natural_language_explanation(explanation, optimal_threshold)
print(f"\n📝 Natural Language Explanation:")
print(f"   {nl_explanation}")


Testing: Extreme Risk Customer

Customer 3 Analysis
Churn Probability: 70.9%
Prediction: ⚠️ HIGH RISK

🔴 Top factors INCREASING churn risk:
   DaysActive: 0.00 (SHAP: +0.5485)
   UniqueProducts: 4.00 (SHAP: +0.3118)
   Monetary: 56.25 (SHAP: +0.0961)

🟢 Top factors DECREASING churn risk:
   AvgQuantity: 9.75 (SHAP: -0.0702)
   OrdersPerDay: 1.00 (SHAP: -0.0302)

📝 Natural Language Explanation:
   This customer has a extreme churn risk (70.9%). The primary driver is their DaysActive (value: 0.0), which increases churn likelihood. Additionally, their UniqueProducts (value: 4.0) contributes to this risk.


**Limitation noted: one-time buyers flagged as churn risk (partially addressed in Day 3)**

Some customers flagged HIGH RISK (e.g. Customer 654, DaysActive=1.00, Frequency=1) are one-time/burst buyers rather than lapsed regulars, and the model doesn't distinguish between the two — both get flagged and would receive the same win-back treatment as-is.

Day 3's segmentation confirmed this is real: the new Extreme Risk tier (probability ≥0.70) is dominated by one-time buyers (median Frequency = 1, median DaysActive = 0), and carries the lowest revenue at risk of any tier despite the highest churn probability — confirming these customers are weak win-back candidates.

However, this only isolates the *most extreme* cases. High Risk (0.5-0.7) still contains a substantial share of one-time buyers too (75th percentile Frequency = 2, meaning over half the tier has Frequency ≤ 2) mixed in with genuinely tenured, higher-value customers. Customer 654 itself is a good example — a clear one-time buyer, but landed in High Risk, not Extreme Risk. Full separation would need a dedicated Frequency-based segmentation layer, not just tighter probability tiers — noted as a possible future enhancement, not built here.